# Assignment 5: Experiment Tracking and Tuning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/neural-architectures-and-representation-learning-course/blob/main/weeks/10/Assignment_05_Experiment_Tracking_Tuning.ipynb)

**Course:** Neural Architectures and Representation Learning  
**Related notebook:** `Week_10_Practical_Deep_Learning_Systems.ipynb`

## Task

You will run and compare practical deep-learning experiments on a small representation model.

The goal is not to find a magic best model. The goal is to show evidence that you can manage experiments responsibly.

You will submit:

1. A baseline run.
2. At least four tracked controlled experiments.
3. A small hyperparameter-search table.
4. Run-comparison plots.
5. A final model decision based on validation evidence.
6. A short freeze-vs-fine-tune workflow decision.
7. A reproducibility checklist.

## Grading rubric

| Criterion | Points |
|----------|--------|
| Correctly runs and logs the baseline | 10 |
| Performs at least four controlled tracked experiments | 25 |
| Compares validation/test metrics and histories clearly | 20 |
| Runs and interprets a small hyperparameter search | 15 |
| Makes a justified final model decision | 15 |
| Explains freeze/fine-tune workflow choice | 10 |
| Clear, reproducible notebook | 5 |

Total: 100 points.

---

## Environment

This notebook uses `torch`, `numpy`, `matplotlib`, and `scikit-learn`. CPU is enough.

## Optional references

| Topic | Resource | Why it helps |
|---|---|---|
| Experiment tracking | [MLflow Tracking docs](https://mlflow.org/docs/latest/ml/tracking/) | Standard vocabulary for runs, parameters, metrics, and artifacts. |
| Dashboards | [Weights & Biases docs](https://docs.wandb.ai/) | Shows what tracked runs look like in a production tool. |
| Local-first tracking | [Hugging Face Trackio docs](https://huggingface.co/docs/trackio/index) | Optional modern tracker with local dashboards and Hugging Face Spaces sharing. |
| Hyperparameter tuning | [Optuna docs](https://optuna.readthedocs.io/) | Official reference for automated hyperparameter optimization. |
| Fine-tuning | [Hugging Face fine-tuning guide](https://huggingface.co/docs/transformers/en/training) | Bridge from this assignment to pretrained-model workflows. |

In [ ]:
import copy
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    plt.rc("axes", grid=True)

%matplotlib inline

print("torch:", torch.__version__)
print("numpy:", np.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(10)

---

## Fixed dataset and utilities

Do not change these utility cells unless you clearly explain why. Your experiments should change configurations, not the evaluation split.

In [ ]:
digits = load_digits()
X = digits.data.astype("float32") / 16.0
y = digits.target.astype("int64")

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=10, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=10, stratify=y_train_full
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_val = scaler.transform(X_val).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

splits = {
    "train": (torch.tensor(X_train), torch.tensor(y_train)),
    "val": (torch.tensor(X_val), torch.tensor(y_val)),
    "test": (torch.tensor(X_test), torch.tensor(y_test)),
}

print("train / val / test:", len(y_train), len(y_val), len(y_test))

class RepresentationMLP(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=64, rep_dim=16, output_dim=10, dropout=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, rep_dim),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(rep_dim, output_dim)

    def forward(self, x):
        return self.classifier(self.features(x))


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def batch_iter(X, y, batch_size=128, shuffle=True, seed=0):
    n = len(y)
    indices = np.arange(n)
    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(indices)
    for start in range(0, n, batch_size):
        idx = indices[start:start + batch_size]
        yield X[idx].to(device), y[idx].to(device)

@torch.no_grad()
def evaluate(model, X, y):
    model.eval()
    losses, correct, total = [], 0, 0
    for xb, yb in batch_iter(X, y, batch_size=256, shuffle=False):
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        pred = logits.argmax(dim=1)
        losses.append(loss.item() * len(yb))
        correct += int((pred == yb).sum())
        total += len(yb)
    return {"loss": float(sum(losses) / total), "accuracy": float(correct / total)}


def train_model(model, train_split, val_split, epochs=24, lr=0.01, weight_decay=0.0, batch_size=128, seed=10):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    Xtr, ytr = train_split
    Xv, yv = val_split
    history = {"train_acc": [], "val_acc": [], "train_loss": [], "val_loss": []}
    best_state, best_val = None, -1.0
    for epoch in range(epochs):
        model.train()
        for xb, yb in batch_iter(Xtr, ytr, batch_size=batch_size, shuffle=True, seed=seed + epoch):
            optimizer.zero_grad()
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            optimizer.step()
        train_metrics = evaluate(model, Xtr, ytr)
        val_metrics = evaluate(model, Xv, yv)
        history["train_acc"].append(train_metrics["accuracy"])
        history["val_acc"].append(val_metrics["accuracy"])
        history["train_loss"].append(train_metrics["loss"])
        history["val_loss"].append(val_metrics["loss"])
        if val_metrics["accuracy"] > best_val:
            best_val = val_metrics["accuracy"]
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, history

In [ ]:
class RunTracker:
    def __init__(self, name):
        self.name = name
        self.runs = []

    def log_run(self, name, config, metrics, history, artifacts=None):
        run = {
            "run_id": f"run_{len(self.runs):03d}",
            "name": name,
            "config": dict(config),
            "metrics": dict(metrics),
            "history": history,
            "artifacts": artifacts or {},
        }
        self.runs.append(run)
        return run

    def table(self):
        rows = []
        for run in self.runs:
            row = {"run_id": run["run_id"], "name": run["name"]}
            row.update(run["config"])
            row.update(run["metrics"])
            rows.append(row)
        return rows

    def best(self, metric="val_accuracy"):
        return max(self.runs, key=lambda r: r["metrics"][metric])


def print_table(rows, columns):
    widths = {c: max(len(c), max(len(f"{r.get(c, '')}") for r in rows)) for c in columns}
    print(" | ".join(c.ljust(widths[c]) for c in columns))
    print("-+-".join("-" * widths[c] for c in columns))
    for row in rows:
        print(" | ".join(f"{row.get(c, '')}".ljust(widths[c]) for c in columns))


def plot_histories(tracker, title):
    plt.figure(figsize=(8, 4))
    for run in tracker.runs:
        plt.plot(run["history"]["val_acc"], label=run["name"])
    plt.xlabel("epoch")
    plt.ylabel("validation accuracy")
    plt.title(title)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


def plot_final_bars(tracker, title="Final comparison"):
    rows = tracker.table()
    names = [r["name"] for r in rows]
    val = [r["val_accuracy"] for r in rows]
    test = [r["test_accuracy"] for r in rows]
    x = np.arange(len(names))
    plt.figure(figsize=(max(7, len(names) * 0.8), 4))
    plt.bar(x - 0.18, val, width=0.36, label="validation")
    plt.bar(x + 0.18, test, width=0.36, label="test")
    plt.xticks(x, names, rotation=30, ha="right")
    plt.ylim(0.75, 1.01)
    plt.ylabel("accuracy")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


def run_experiment(name, config, tracker, verbose=True):
    set_seed(config.get("seed", 10))
    start = time.time()
    model = RepresentationMLP(
        hidden_dim=config["hidden_dim"],
        rep_dim=config["rep_dim"],
        dropout=config.get("dropout", 0.0),
    )
    params = count_parameters(model)
    model, history = train_model(
        model,
        splits["train"],
        splits["val"],
        epochs=config["epochs"],
        lr=config["lr"],
        weight_decay=config.get("weight_decay", 0.0),
        batch_size=config.get("batch_size", 128),
        seed=config.get("seed", 10),
    )
    val = evaluate(model, *splits["val"])
    test = evaluate(model, *splits["test"])
    metrics = {
        "val_accuracy": round(val["accuracy"], 4),
        "test_accuracy": round(test["accuracy"], 4),
        "parameters": params,
        "seconds": round(time.time() - start, 2),
    }
    run = tracker.log_run(name, config, metrics, history, artifacts={"model": model})
    if verbose:
        print(name, metrics)
    return model, run

tracker = RunTracker("assignment5_digits")
columns = ["run_id", "name", "hidden_dim", "rep_dim", "lr", "dropout", "weight_decay", "val_accuracy", "test_accuracy", "parameters"]

---

## 1. Baseline run

Run this first. Do not edit the baseline.

In [ ]:
baseline_config = {
    "hidden_dim": 48,
    "rep_dim": 12,
    "dropout": 0.0,
    "lr": 0.01,
    "weight_decay": 0.0,
    "batch_size": 128,
    "epochs": 24,
    "seed": 10,
}

baseline_model, baseline_run = run_experiment("baseline", baseline_config, tracker)
print_table(tracker.table(), columns)
plot_histories(tracker, "Baseline validation curve")

## 2. Controlled experiments

Run at least four controlled experiments. Change one or two values at a time, and explain why.

The starter configs below are examples. You may edit them.

In [ ]:
experiment_configs = [
    ("exp_1_wider", {**baseline_config, "hidden_dim": 96, "seed": 31}),
    ("exp_2_smaller_rep", {**baseline_config, "rep_dim": 6, "seed": 32}),
    ("exp_3_dropout", {**baseline_config, "dropout": 0.2, "weight_decay": 0.001, "seed": 33}),
    ("exp_4_low_lr", {**baseline_config, "lr": 0.003, "epochs": 30, "seed": 34}),
]

for name, config in experiment_configs:
    run_experiment(name, config, tracker)

print_table(tracker.table(), columns)
plot_histories(tracker, "Controlled experiment validation curves")
plot_final_bars(tracker, "Baseline and controlled experiments")

best_controlled = tracker.best("val_accuracy")
print("Best controlled run:", best_controlled["name"], best_controlled["metrics"])

### Controlled-experiment interpretation

Write 5-8 sentences:

1. Which change helped most?
2. Which change hurt or did not matter?
3. Did validation and test agree?
4. Did a larger model always help?
5. Which run would you keep for the next stage?

---

## 3. Small hyperparameter search

This is a tiny random search. It is not meant to be exhaustive. It should show that search results are tracked evidence, not magic.

For this small model, a full grid search would also be possible if the grid had only a few values. For larger models, you should usually avoid changing many things at once. Use a controlled ladder instead: pick one parameter to vary, keep the best stable setting, then move to the next parameter.

In [ ]:
def sample_config(trial_id, seed=200):
    rng = np.random.default_rng(seed + trial_id)
    return {
        "hidden_dim": int(rng.choice([32, 48, 64, 96, 128])),
        "rep_dim": int(rng.choice([6, 10, 12, 16, 24, 32])),
        "dropout": float(rng.choice([0.0, 0.1, 0.2, 0.3])),
        "lr": float(rng.choice([0.001, 0.003, 0.006, 0.01, 0.02])),
        "weight_decay": float(rng.choice([0.0, 0.0005, 0.001, 0.003])),
        "batch_size": 128,
        "epochs": 18,
        "seed": int(200 + trial_id),
    }

search_tracker = RunTracker("assignment5_search")

# TODO: use at least 6 trials. You may increase this if runtime is acceptable.
for trial_id in range(6):
    config = sample_config(trial_id)
    run_experiment(f"trial_{trial_id}", config, search_tracker, verbose=False)

print_table(search_tracker.table(), columns)
plot_histories(search_tracker, "Search validation curves")
plot_final_bars(search_tracker, "Search final metrics")
print("Best search trial:", search_tracker.best("val_accuracy")["name"], search_tracker.best("val_accuracy")["config"], search_tracker.best("val_accuracy")["metrics"])

### Search interpretation

Write 4-6 sentences:

1. What was the best search configuration?
2. Did it beat your best controlled run?
3. Was the difference large enough to trust?
4. Which hyperparameter seemed most important?
5. What would you search next?
6. Optional: how would this section change if you used Optuna trials instead of the provided random search?
7. If this model were 100x more expensive to train, would you use grid search, random/Optuna search, or a controlled ladder? Why?

---

## 4. Final decision

Choose one final model based on validation evidence. You may choose from the controlled runs or search runs.

In [ ]:
all_candidates = tracker.runs + search_tracker.runs
# Choose by validation first; if validation is tied, prefer the smaller model.
best_candidate = sorted(
    all_candidates,
    key=lambda r: (-r["metrics"]["val_accuracy"], r["metrics"]["parameters"]),
)[0]

print("Chosen by validation accuracy, tie-broken by parameter count:")
print(best_candidate["run_id"], best_candidate["name"])
print("config:", best_candidate["config"])
print("metrics:", best_candidate["metrics"])

# TODO: you may override this if you prefer a simpler or more stable model.
final_choice_name = best_candidate["name"]
print("final_choice_name =", final_choice_name)

### Final model justification

Write 6-8 sentences:

1. Which model did you choose?
2. What validation evidence supports it?
3. What test result did it get?
4. Was it simpler or larger than the baseline?
5. What are the main risks of your choice?
6. What would you re-run with another seed before trusting it?

---

## 5. Freeze vs fine-tune workflow decision

This section simulates a small-data transfer decision. You do not need a large pretrained model to explain the workflow.

In [ ]:
# Train one source model on the 10-class task.
source_config = {**baseline_config, "hidden_dim": 96, "rep_dim": 24, "epochs": 28, "seed": 90}
source_tracker = RunTracker("source")
source_model, _ = run_experiment("source_10_class", source_config, source_tracker, verbose=False)

# Build a small even-vs-odd target task.
y_binary = (y % 2).astype("int64")
X_train_full_b, X_test_b, y_train_full_b, y_test_b = train_test_split(
    X, y_binary, test_size=0.20, random_state=20, stratify=y_binary
)
X_train_b, X_val_b, y_train_b, y_val_b = train_test_split(
    X_train_full_b, y_train_full_b, train_size=120, random_state=20, stratify=y_train_full_b
)

# Use the source-task scaler so copied source representations see the same preprocessing.
X_train_b = scaler.transform(X_train_b).astype("float32")
X_val_b = scaler.transform(X_val_b).astype("float32")
X_test_b = scaler.transform(X_test_b).astype("float32")

binary_splits = {
    "train": (torch.tensor(X_train_b), torch.tensor(y_train_b)),
    "val": (torch.tensor(X_val_b), torch.tensor(y_val_b)),
    "test": (torch.tensor(X_test_b), torch.tensor(y_test_b)),
}

class BinaryRepresentationMLP(RepresentationMLP):
    def __init__(self):
        super().__init__(hidden_dim=96, rep_dim=24, output_dim=2, dropout=0.0)


def make_binary_model(source=None, freeze_features=False, seed=0):
    set_seed(seed)
    model = BinaryRepresentationMLP()
    if source is not None:
        model.features.load_state_dict(copy.deepcopy(source.features.state_dict()))
    if freeze_features:
        for p in model.features.parameters():
            p.requires_grad = False
    return model


def run_binary(name, model, config, tracker):
    start = time.time()
    model, history = train_model(
        model,
        binary_splits["train"],
        binary_splits["val"],
        epochs=config["epochs"],
        lr=config["lr"],
        batch_size=64,
        seed=config["seed"],
    )
    val = evaluate(model, *binary_splits["val"])
    test = evaluate(model, *binary_splits["test"])
    metrics = {
        "val_accuracy": round(val["accuracy"], 4),
        "test_accuracy": round(test["accuracy"], 4),
        "parameters": count_parameters(model),
        "seconds": round(time.time() - start, 2),
    }
    return tracker.log_run(name, config, metrics, history)

transfer_tracker = RunTracker("transfer")
transfer_configs = [
    ("scratch", make_binary_model(None, False, 101), {"strategy": "scratch", "epochs": 30, "lr": 0.01, "seed": 101}),
    ("frozen_head", make_binary_model(source_model, True, 102), {"strategy": "freeze_features", "epochs": 30, "lr": 0.01, "seed": 102}),
    ("fine_tune", make_binary_model(source_model, False, 103), {"strategy": "fine_tune", "epochs": 30, "lr": 0.003, "seed": 103}),
]

for name, model, config in transfer_configs:
    run_binary(name, model, config, transfer_tracker)

transfer_columns = ["run_id", "name", "strategy", "lr", "val_accuracy", "test_accuracy", "parameters"]
print_table(transfer_tracker.table(), transfer_columns)
plot_histories(transfer_tracker, "Freeze vs fine-tune validation curves")
plot_final_bars(transfer_tracker, "Transfer strategy comparison")

### Transfer workflow interpretation

Write 5-7 sentences:

1. Which strategy had the best validation score?
2. Which strategy trained the fewest parameters?
3. Did the source representation appear useful?
4. Which strategy would you choose for a small real dataset?
5. What would you check before trusting the result?

---

## 6. Reproducibility checklist

Fill this in before submission:

- [ ] I kept the train/validation/test split fixed.
- [ ] I logged all configs I compared.
- [ ] I compared validation before looking at test as final evidence.
- [ ] I included plots or tables for run comparison.
- [ ] I explained why I chose the final model.
- [ ] I mentioned at least one limitation or risk.
- [ ] I can explain what would change in a real MLflow/W&B/Trackio/Optuna workflow.